# MBDoE 2026 Lot 2: integrated data preview and QC

## tl;dr

This notebook reconstructs the three synthetic-must Lot 2 fermentations from raw laboratory and process files. It preserves raw analytical values, creates separately named model-safe values, and makes every timing or identifier repair auditable. The executed outputs below determine which observations can enter the next calibration and estimability comparison.

## Context & Methods

### Key assumptions

1. The first Oculyze sample defines $t=0$ for each reactor.
2. Oculyze concentration is total cells in million cells/mL. Using 30 pg/cell,

$$X_{\mathrm{total}}[\mathrm{g/L}] = C[10^6\,\mathrm{cells/mL}]\times 0.03.$$

Viable and dead biomass are

$$X = X_{\mathrm{total}}\frac{V}{100},\qquad X_d=X_{\mathrm{total}}-X.$$

3. Ethanol is recomputed from manual entries using $E[\mathrm{g/L}]=7.89\,E[\%\,v/v]$.
4. A measured-ethanol sample removes 50 mL; another sample with analytical evidence removes 5 mL. A planned row without any measurement evidence removes 0 mL.
5. CO2 flow is normalized to the initial 2 L volume with

$$q_{\mathrm{CO_2},0}=q_{\mathrm{CO_2}}\frac{V_0}{V(t)}.$$

Protocol events and inferred times are labeled; they are not presented as direct observations.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
while ROOT.name != "pyomo-doe" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "pyomo-doe":
    raise RuntimeError("Run this notebook from inside the pyomo-doe repository")

import sys
FERMENTATION_MODEL = ROOT / "fermentation_model"
if str(FERMENTATION_MODEL) not in sys.path:
    sys.path.insert(0, str(FERMENTATION_MODEL))

from laboratory_2026 import run_lot2_data_preview as analysis

summary = analysis.run_pipeline()
summary

## Data

In [ ]:
processed = analysis.PROCESSED_DIR
qc = pd.read_csv(processed / "lot2_data_qc_summary.csv")
samples = pd.read_csv(processed / "lot2_samples_integrated.csv")
inputs = pd.read_csv(processed / "lot2_operational_inputs.csv")
repairs = pd.read_csv(processed / "lot2_oculyze_repairs.csv")
display(qc)
display(inputs)
display(repairs)

### Data-quality rules

- Raw negative analytical values remain available. Only columns ending in `for_model` are censored at zero.
- Oculyze identifiers are repaired only when the row sequence makes the typo unambiguous; the raw identifier remains beside the repaired one.
- Missing collection times are estimated from Y15 analysis time minus the process-specific median analytical lag.
- The ethanol mass-balance flag is a screening test, not an automatic data correction.

In [ ]:
columns = [
    "process", "sample_id", "sample_sequence", "sample_datetime_used", "t_h",
    "sample_time_source", "glucose_g_l", "fructose_g_l_raw", "yan_for_model_physical_mg_l",
    "ethanol_g_l_for_model", "ethanol_use_for_model", "glycerol_g_l", "x_viable_g_l",
    "x_dead_g_l", "reactor_volume_after_sample_ml"
]
display(samples[columns].head(24))

## Results

In [ ]:
for process in ("F1", "F2", "F3"):
    display(Image(filename=str(analysis.FIGURE_DIR / f"lot2_{process}_process_preview.png")))

## Takeaways

In [ ]:
display(Markdown((analysis.RESULTS_DIR / "lot2_data_quality_report.md").read_text(encoding="utf-8")))